In [24]:
from pathlib import Path
import shutil
import json

ICPR_IMAGE_TEST_PATH = Path("../../datasets/ICPR2022/ICPR2022_CHARTINFO_UB_UNITEC_PMC_TEST_v2.1/chart_images")
ICPR_ANNOTATION_TEST_PATH = Path("../../datasets/ICPR2022/ICPR2022_CHARTINFO_UB_UNITEC_PMC_TEST_v2.1/final_full_GT")
TRAINING_SET_PATH = Path("../../training_data/ICPR2022_test/")

In [32]:
def get_filenames_for_label(annotation_path, findlabel):
    '''scorre le directory contenenti le annotazioni del test set di ICPR2022 e leggi i file json
    fornisce i filename delle immagini di una classe di grafici/immagini'''
    img_names = []
    for ann_dir in annotation_path.iterdir():
        ann_dir_path = ann_dir / "annotations_JSON"
        for ann in ann_dir_path.iterdir():
            with open(ann, "r") as f:
                data = json.load(f)
                label = data.get("task1").get("output").get("chart_type")
                if findlabel == label:
                        img_name = ann.with_suffix(".jpg").name
                        img_names.append(img_name)
    return img_names
    

def find_images_recursively(search_path, img_names):
    search_path = Path(search_path)
    names_to_find = set(img_names)
    found_paths = []

    print(f"Inizio ricerca ricorsiva in: {search_path}...")

    for file_path in search_path.rglob("*"):
        # Verifichiamo se è un file e se il suo nome è nella nostra lista
        if file_path.is_file() and file_path.name in names_to_find:
            found_paths.append(file_path)
    print(f"Ricerca completata. Trovati {len(found_paths)} file su {len(img_names)} richiesti.")
    return found_paths

def copy_file_to(from_path, to_dir, label):
    target_dir = Path(to_dir) / label
    target_dir.mkdir(parents=True, exist_ok=True)
    for imgpath in from_path:
        to_path = target_dir / imgpath.name
        shutil.copy2(imgpath, to_path)

In [42]:
labels = ["area", "heatmap", "horizontal bar", "horizontal interval", "line", "manhattan", "map", "pie",
            "scatter", "scatter-line", "surface", "venn", "vertical bar", "vertical box", "vertical interval"]

In [43]:
for label in labels:
    print(label)
    img_filenames = get_filenames_for_label(ICPR_ANNOTATION_TEST_PATH, label)
    imgs_fullpath = find_images_recursively(ICPR_IMAGE_TEST_PATH, img_filenames)
    copy_file_to(imgs_fullpath, TRAINING_SET_PATH, label)

area
Inizio ricerca ricorsiva in: ../../datasets/ICPR2022/ICPR2022_CHARTINFO_UB_UNITEC_PMC_TEST_v2.1/chart_images...
Ricerca completata. Trovati 136 file su 136 richiesti.
heatmap
Inizio ricerca ricorsiva in: ../../datasets/ICPR2022/ICPR2022_CHARTINFO_UB_UNITEC_PMC_TEST_v2.1/chart_images...
Ricerca completata. Trovati 180 file su 180 richiesti.
horizontal bar
Inizio ricerca ricorsiva in: ../../datasets/ICPR2022/ICPR2022_CHARTINFO_UB_UNITEC_PMC_TEST_v2.1/chart_images...
Ricerca completata. Trovati 634 file su 634 richiesti.
horizontal interval
Inizio ricerca ricorsiva in: ../../datasets/ICPR2022/ICPR2022_CHARTINFO_UB_UNITEC_PMC_TEST_v2.1/chart_images...
Ricerca completata. Trovati 430 file su 430 richiesti.
line
Inizio ricerca ricorsiva in: ../../datasets/ICPR2022/ICPR2022_CHARTINFO_UB_UNITEC_PMC_TEST_v2.1/chart_images...
Ricerca completata. Trovati 3399 file su 3399 richiesti.
manhattan
Inizio ricerca ricorsiva in: ../../datasets/ICPR2022/ICPR2022_CHARTINFO_UB_UNITEC_PMC_TEST_v2.1/char

In [44]:
from pathlib import Path

def check_duplicate_filenames(root_path):
    root = Path(root_path)
    # Dizionario che mappa {nome_file: [lista_percorsi_dove_appare]}
    files_map = {}

    print(f"Analisi in corso in: {root.absolute()}")

    # Scansione ricorsiva di tutti i file
    for file_path in root.rglob("*"):
        if file_path.is_file():
            name = file_path.name
            if name not in files_map:
                files_map[name] = []
            files_map[name].append(file_path)

    # Identificazione dei duplicati
    duplicates = {name: paths for name, paths in files_map.items() if len(paths) > 1}

    if not duplicates:
        print("Ottimo! Non sono stati trovati file con nomi duplicati.")
    else:
        print(f"ATTENZIONE: Trovati {len(duplicates)} nomi di file duplicati!\n")
        for name, paths in duplicates.items():
            print(f"File: {name}")
            for p in paths:
                print(f"  - In: {p.parent}")
            print("-" * 20)
            
    return duplicates

duplicati = check_duplicate_filenames("../../training_data")

Analisi in corso in: /home/giorgiomelch/projects/chart-classifier_p/data_management/notebooks/create_trainingset/../../training_data
Ottimo! Non sono stati trovati file con nomi duplicati.
